In [17]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler


df = pd.read_csv(
    "../data/preprocessed_data/forecasting_model_dataset.csv"
)

df["date"] = pd.to_datetime(
    df["date"]
)

df = df.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)


print(df.shape)

print(
    df["date"].min(),
    "→",
    df["date"].max()
)

display(
    df.groupby("company")[
        "date"
    ].agg(
        ["min", "max", "count"]
    )
)

(4924, 42)
2022-09-01 00:00:00 → 2026-09-11 00:00:00


,min,max,count
company,,,
CIMB,2022-09-01,2026-09-11,985
Maybank,2022-09-01,2026-09-11,985
Petronas Chemicals,2022-09-01,2026-09-11,985
Public Bank,2022-09-01,2026-09-11,984
Tenaga Nasional,2022-09-01,2026-09-11,985


In [18]:
# ============================================================
# 2. FEATURES AND TARGET
# ============================================================

feature_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",

    "daily_return",
    "intraday_return",
    "daily_range",
    "volume_change",

    "rsi_14",

    "sma_20",
    "sma_50",
    "sma_200",

    "ema_12",
    "ema_26",

    "macd",
    "macd_signal",
    "macd_histogram",

    "atr_14",

    "bollinger_width",

    "volatility_20",
    "momentum_10",

    "news_count",
    "sentiment_mean",
    "sentiment_max",
    "sentiment_min",

    "positive_score_mean",
    "negative_score_mean",
    "neutral_score_mean",

    "positive_count",
    "negative_count",
    "neutral_count"
]


target_column = "target_return_1d"


print(
    "Number of features:",
    len(feature_columns)
)


print(
    "Missing values in features:",
    df[feature_columns].isnull().sum().sum()
)


print(
    "Missing target:",
    df[target_column].isnull().sum()
)

Number of features: 32
Missing values in features: 0
Missing target: 0


In [19]:
# ============================================================
# CLEAN INVALID VALUES
# ============================================================

# Afficher les colonnes contenant +inf ou -inf
inf_counts = np.isinf(df[feature_columns]).sum()

print("Infinite values:")
print(
    inf_counts[
        inf_counts > 0
    ]
)

# Transformer +inf et -inf en NaN
df[feature_columns] = df[feature_columns].replace(
    [np.inf, -np.inf],
    np.nan
)

df[target_column] = df[target_column].replace(
    [np.inf, -np.inf],
    np.nan
)

# Supprimer uniquement les lignes devenues invalides
df = (
    df
    .dropna(
        subset=feature_columns + [target_column]
    )
    .sort_values(
        ["ticker", "date"]
    )
    .reset_index(drop=True)
)

# Vérification finale
print(
    "Remaining NaN:",
    df[
        feature_columns + [target_column]
    ].isna().sum().sum()
)

print(
    "Remaining infinity:",
    np.isinf(
        df[
            feature_columns + [target_column]
        ]
    ).sum().sum()
)

print(
    "Dataset shape after cleaning:",
    df.shape
)

Infinite values:
volume_change    6
dtype: int64
Remaining NaN: 0
Remaining infinity: 0
Dataset shape after cleaning: (4918, 42)


In [20]:
# ============================================================
# 3. CHRONOLOGICAL SPLIT
# ============================================================

unique_dates = np.array(
    sorted(
        df["date"].unique()
    )
)


n_dates = len(
    unique_dates
)


train_cut = int(
    n_dates * 0.70
)

val_cut = int(
    n_dates * 0.85
)


print(
    "Number of unique trading dates:",
    n_dates
)

Number of unique trading dates: 984


In [21]:
# ------------------------------------------------------------
# One-day embargo between splits
# ------------------------------------------------------------

train_end = pd.Timestamp(
    unique_dates[
        train_cut - 2
    ]
)


val_start = pd.Timestamp(
    unique_dates[
        train_cut
    ]
)


val_end = pd.Timestamp(
    unique_dates[
        val_cut - 2
    ]
)


test_start = pd.Timestamp(
    unique_dates[
        val_cut
    ]
)


print(
    "TRAIN:",
    df["date"].min(),
    "→",
    train_end
)


print(
    "VALIDATION:",
    val_start,
    "→",
    val_end
)


print(
    "TEST:",
    test_start,
    "→",
    df["date"].max()
)

TRAIN: 2022-09-01 00:00:00 → 2025-06-24 00:00:00
VALIDATION: 2025-06-26 00:00:00 → 2026-01-29 00:00:00
TEST: 2026-02-03 00:00:00 → 2026-09-11 00:00:00


In [22]:
train_rows = df[
    df["date"] <= train_end
]

val_rows = df[
    (df["date"] >= val_start)
    &
    (df["date"] <= val_end)
]

test_rows = df[
    df["date"] >= test_start
]


print(
    "Train rows:",
    len(train_rows)
)

print(
    "Validation rows:",
    len(val_rows)
)

print(
    "Test rows:",
    len(test_rows)
)

Train rows: 3433
Validation rows: 735
Test rows: 740


In [23]:
print("\nTrain companies")
print(
    train_rows[
        "company"
    ].value_counts()
)


print("\nValidation companies")
print(
    val_rows[
        "company"
    ].value_counts()
)


print("\nTest companies")
print(
    test_rows[
        "company"
    ].value_counts()
)


Train companies
company
CIMB                  687
Maybank               687
Petronas Chemicals    687
Tenaga Nasional       687
Public Bank           685
Name: count, dtype: int64

Validation companies
company
CIMB                  147
Maybank               147
Public Bank           147
Petronas Chemicals    147
Tenaga Nasional       147
Name: count, dtype: int64

Test companies
company
CIMB                  148
Maybank               148
Public Bank           148
Petronas Chemicals    148
Tenaga Nasional       148
Name: count, dtype: int64


In [24]:
from sklearn.preprocessing import StandardScaler

In [25]:
# ============================================================
# 4. FEATURE SCALING
# ============================================================

scaler = StandardScaler()
scaler.fit(
    train_rows[
        feature_columns
    ]
)

StandardScaler()

In [26]:
scaled_df = df.copy()


scaled_df[
    feature_columns
] = scaler.transform(
    df[
        feature_columns
    ]
)

In [27]:
display(
    scaled_df[
        feature_columns
    ].head()
)

,open,high,low,close,volume,daily_return,intraday_return,daily_range,volume_change,rsi_14,...,news_count,sentiment_mean,sentiment_max,sentiment_min,positive_score_mean,negative_score_mean,neutral_score_mean,positive_count,negative_count,neutral_count
0,-0.990058,-0.946928,-0.976604,-0.945366,1.173514,1.594107,2.360793,1.901316,-0.069118,0.869954,...,5.040604,6.231973,6.231973,6.231973,6.943305,0.174162,0.161012,6.353066,-0.082127,-0.083906
1,-0.949257,-0.946929,-0.935554,-0.939548,0.143019,0.270655,0.453136,-0.611822,-0.749715,0.976329,...,-0.198389,-0.100222,-0.100222,-0.100222,-0.162965,-0.096396,-0.115101,-0.157404,-0.082127,-0.083906
2,-0.940514,-0.946928,-0.935554,-0.945366,-0.382322,-0.302122,-0.314777,-0.616550,-0.715733,0.788427,...,-0.198389,-0.100222,-0.100222,-0.100222,-0.162965,-0.096396,-0.115101,-0.157404,-0.082127,-0.083906
3,-0.946342,-0.949822,-0.944350,-0.951185,-0.449654,-0.303174,-0.315895,-0.267374,-0.335893,0.599906,...,-0.198389,-0.100222,-0.100222,-0.100222,-0.162965,-0.096396,-0.115101,-0.157404,-0.082127,-0.083906
4,-0.952171,-0.961395,-0.959011,-0.968640,-0.285885,-0.880157,-0.933819,-0.089294,0.066001,0.072836,...,-0.198389,-0.100222,-0.100222,-0.100222,-0.162965,-0.096396,-0.115101,-0.157404,-0.082127,-0.083906


In [28]:
print(
    "TRAIN means after scaling:"
)

print(
    scaled_df[
        scaled_df["date"] <= train_end
    ][
        feature_columns
    ]
    .mean()
    .head(10)
)

TRAIN means after scaling:
open               8.278972e-17
high               3.146009e-16
low                4.139486e-16
close             -6.623177e-17
volume             4.967383e-17
daily_return      -9.313843e-18
intraday_return    1.552307e-18
daily_range       -1.655794e-16
volume_change     -1.034871e-17
rsi_14            -1.821374e-16
dtype: float64


In [29]:
print(
    "\nTRAIN standard deviations:"
)

print(
    scaled_df[
        scaled_df["date"] <= train_end
    ][
        feature_columns
    ]
    .std()
    .head(10)
)


TRAIN standard deviations:
open               1.000146
high               1.000146
low                1.000146
close              1.000146
volume             1.000146
daily_return       1.000146
intraday_return    1.000146
daily_range        1.000146
volume_change      1.000146
rsi_14             1.000146
dtype: float64


In [30]:
# ============================================================
# 5. SEQUENCE CREATION
# ============================================================

WINDOW_SIZE = 60

def create_sequences(
    data,
    feature_columns,
    target_column,
    window_size,
    train_end,
    val_start,
    val_end,
    test_start
):

    X_train = []
    y_train = []

    X_val = []
    y_val = []

    X_test = []
    y_test = []


    train_metadata = []
    val_metadata = []
    test_metadata = []


    # Important :
    # créer les séquences séparément
    # pour chaque entreprise
    for ticker, group in data.groupby(
        "ticker"
    ):

        group = (
            group
            .sort_values("date")
            .reset_index(drop=True)
        )


        features = (
            group[
                feature_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        targets = (
            group[
                target_column
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        dates = (
            group[
                "date"
            ]
            .to_numpy()
        )


        companies = (
            group[
                "company"
            ]
            .to_numpy()
        )


        # --------------------------------------------
        # Sliding window
        # --------------------------------------------

        for i in range(
            window_size - 1,
            len(group)
        ):

            start = (
                i
                -
                window_size
                +
                1
            )


            X_window = features[
                start:i + 1
            ]


            y_target = targets[
                i
            ]


            end_date = pd.Timestamp(
                dates[i]
            )


            metadata = {
                "ticker":
                    ticker,

                "company":
                    companies[i],

                "date":
                    end_date
            }


            # ========================================
            # TRAIN
            # ========================================

            if end_date <= train_end:

                X_train.append(
                    X_window
                )

                y_train.append(
                    y_target
                )

                train_metadata.append(
                    metadata
                )


            # ========================================
            # VALIDATION
            # ========================================

            elif (
                end_date >= val_start
                and
                end_date <= val_end
            ):

                X_val.append(
                    X_window
                )

                y_val.append(
                    y_target
                )

                val_metadata.append(
                    metadata
                )


            # ========================================
            # TEST
            # ========================================

            elif end_date >= test_start:

                X_test.append(
                    X_window
                )

                y_test.append(
                    y_target
                )

                test_metadata.append(
                    metadata
                )


    return (
        np.array(X_train),
        np.array(y_train),

        np.array(X_val),
        np.array(y_val),

        np.array(X_test),
        np.array(y_test),

        pd.DataFrame(
            train_metadata
        ),

        pd.DataFrame(
            val_metadata
        ),

        pd.DataFrame(
            test_metadata
        )
    )

In [31]:
(
    X_train,
    y_train,

    X_val,
    y_val,

    X_test,
    y_test,

    meta_train,
    meta_val,
    meta_test

) = create_sequences(

    data=scaled_df,

    feature_columns=feature_columns,

    target_column=target_column,

    window_size=WINDOW_SIZE,

    train_end=train_end,

    val_start=val_start,

    val_end=val_end,

    test_start=test_start
)

In [32]:
print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)


print(
    "\nX_val:",
    X_val.shape
)

print(
    "y_val:",
    y_val.shape
)


print(
    "\nX_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)

X_train: (3138, 60, 32)
y_train: (3138,)

X_val: (735, 60, 32)
y_val: (735,)

X_test: (740, 60, 32)
y_test: (740,)


In [33]:
print(
    meta_train[
        "company"
    ].value_counts()
)


print(
    meta_val[
        "company"
    ].value_counts()
)


print(
    meta_test[
        "company"
    ].value_counts()
)

company
CIMB                  628
Maybank               628
Petronas Chemicals    628
Tenaga Nasional       628
Public Bank           626
Name: count, dtype: int64
company
CIMB                  147
Maybank               147
Public Bank           147
Petronas Chemicals    147
Tenaga Nasional       147
Name: count, dtype: int64
company
CIMB                  148
Maybank               148
Public Bank           148
Petronas Chemicals    148
Tenaga Nasional       148
Name: count, dtype: int64


In [34]:
# ============================================================
# 6. NAIVE BASELINE
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
baseline_predictions = np.zeros_like(
    y_test
)

In [35]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)


baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)


print(
    "Baseline MAE:",
    baseline_mae
)


print(
    "Baseline RMSE:",
    baseline_rmse
)

Baseline MAE: 1.2977970838546753
Baseline RMSE: 2.262633354514259


In [36]:
def directional_accuracy(
    y_true,
    y_pred
):

    true_direction = (
        y_true > 0
    )

    pred_direction = (
        y_pred > 0
    )

    return np.mean(
        true_direction
        ==
        pred_direction
    )

In [37]:
baseline_direction_accuracy = (
    directional_accuracy(
        y_test,
        baseline_predictions
    )
)


print(
    "Baseline Direction Accuracy:",
    baseline_direction_accuracy
)

Baseline Direction Accuracy: 0.5472972972972973


In [38]:
# ============================================================
# 7. LSTM MODEL
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping


tf.random.set_seed(42)
np.random.seed(42)

In [39]:
n_features = X_train.shape[2]


lstm_model = Sequential([
    LSTM(
        64,
        input_shape=(
            WINDOW_SIZE,
            n_features
        )
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])


lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


lstm_model.summary()

c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,945 (105.25 KB)

 Trainable params: 26,945 (105.25 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [41]:
history = lstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        early_stopping
    ],

    verbose=1
)

Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1.6458 - mae: 0.8303 - val_loss: 2.7356 - val_mae: 1.0217
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 1.6129 - mae: 0.8210 - val_loss: 2.7395 - val_mae: 1.0154
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 1.5920 - mae: 0.8158 - val_loss: 2.7324 - val_mae: 1.0166
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 1.5708 - mae: 0.8111 - val_loss: 2.7517 - val_mae: 1.0198
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.5577 - mae: 0.8075 - val_loss: 2.7632 - val_mae: 1.0191
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 1.5377 - mae: 0.8042 - val_loss: 2.7665 - val_mae: 1.0213
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.5292 - mae: 0.8021 - val_loss: 2.7668 - val_mae: 1.0188
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.5078 - mae: 0.8003 - val_loss: 2.7840 - val_mae: 1.0273
Epoch 9/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - lo

In [42]:
lstm_predictions = (
    lstm_model
    .predict(
        X_test
    )
    .flatten()
)

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


In [43]:
lstm_mae = mean_absolute_error(
    y_test,
    lstm_predictions
)


lstm_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        lstm_predictions
    )
)


lstm_direction_accuracy = (
    directional_accuracy(
        y_test,
        lstm_predictions
    )
)


print(
    "LSTM MAE:",
    lstm_mae
)

print(
    "LSTM RMSE:",
    lstm_rmse
)

print(
    "LSTM Direction Accuracy:",
    lstm_direction_accuracy
)

LSTM MAE: 1.2944080829620361
LSTM RMSE: 2.2721679918345994
LSTM Direction Accuracy: 0.5648648648648649


In [44]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Zero Return",
        "LSTM"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy
    ]
})


display(comparison)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.297797,2.262633,0.547297
1,LSTM,1.294408,2.272168,0.564865


In [45]:
# ============================================================
# 8. BiLSTM MODEL
# ============================================================

from tensorflow.keras.layers import Bidirectional

bilstm_model = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(
            WINDOW_SIZE,
            n_features
        )
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])


bilstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


bilstm_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


bilstm_history = bilstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        bilstm_early_stopping
    ],

    verbose=1
)

Epoch 1/100


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


99/99 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 1.6619 - mae: 0.8393 - val_loss: 2.7824 - val_mae: 1.0230
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 1.6089 - mae: 0.8237 - val_loss: 2.8036 - val_mae: 1.0290
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.5760 - mae: 0.8168 - val_loss: 2.8334 - val_mae: 1.0290
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 1.5570 - mae: 0.8126 - val_loss: 2.8749 - val_mae: 1.0312
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 1.5232 - mae: 0.8036 - val_loss: 2.8719 - val_mae: 1.0353
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.5115 - mae: 0.8009 - val_loss: 2.8765 - val_mae: 1.0313
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.4752 - mae: 0.8005 - val_loss: 2.8842 - val_mae: 1.0342
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.4653 - mae: 0.7964 - val_loss: 2.9335 - val_mae: 1.0391
Epoch 9/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.4344 -

In [46]:
bilstm_predictions = (
    bilstm_model
    .predict(
        X_test
    )
    .flatten()
)


bilstm_mae = mean_absolute_error(
    y_test,
    bilstm_predictions
)


bilstm_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        bilstm_predictions
    )
)


bilstm_direction_accuracy = (
    directional_accuracy(
        y_test,
        bilstm_predictions
    )
)


print(
    "BiLSTM MAE:",
    bilstm_mae
)

print(
    "BiLSTM RMSE:",
    bilstm_rmse
)

print(
    "BiLSTM Direction Accuracy:",
    bilstm_direction_accuracy
)

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
BiLSTM MAE: 1.3163564205169678
BiLSTM RMSE: 2.313080585901624
BiLSTM Direction Accuracy: 0.5027027027027027


In [47]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Zero Return",
        "LSTM",
        "BiLSTM"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae,
        bilstm_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse,
        bilstm_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy,
        bilstm_direction_accuracy
    ]
})


display(comparison)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.297797,2.262633,0.547297
1,LSTM,1.294408,2.272168,0.564865
2,BiLSTM,1.316356,2.313081,0.502703


In [48]:
# ============================================================
# 9. TRANSFORMER MODEL
# ============================================================

from tensorflow.keras import Model, Input

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding
)


# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT_RATE = 0.2


# ------------------------------------------------------------
# Input
# ------------------------------------------------------------

inputs = Input(
    shape=(
        WINDOW_SIZE,
        n_features
    )
)


# ------------------------------------------------------------
# Project features to D_MODEL dimensions
# ------------------------------------------------------------

x = Dense(
    D_MODEL
)(inputs)


# ------------------------------------------------------------
# Positional embedding
# ------------------------------------------------------------

positions = tf.range(
    start=0,
    limit=WINDOW_SIZE,
    delta=1
)


position_embedding = Embedding(
    input_dim=WINDOW_SIZE,
    output_dim=D_MODEL
)(positions)


x = x + position_embedding


# ------------------------------------------------------------
# Multi-Head Self-Attention
# ------------------------------------------------------------

attention_output = MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=D_MODEL // NUM_HEADS
)(
    x,
    x
)


attention_output = Dropout(
    DROPOUT_RATE
)(
    attention_output
)


# Residual connection + normalization
x = LayerNormalization(
    epsilon=1e-6
)(
    x + attention_output
)


# ------------------------------------------------------------
# Feed Forward Network
# ------------------------------------------------------------

ff_output = Dense(
    FF_DIM,
    activation="relu"
)(x)


ff_output = Dropout(
    DROPOUT_RATE
)(
    ff_output
)


ff_output = Dense(
    D_MODEL
)(
    ff_output
)


# Residual connection + normalization
x = LayerNormalization(
    epsilon=1e-6
)(
    x + ff_output
)


# ------------------------------------------------------------
# Convert sequence → single representation
# ------------------------------------------------------------

x = GlobalAveragePooling1D()(
    x
)


x = Dense(
    32,
    activation="relu"
)(
    x
)


x = Dropout(
    DROPOUT_RATE
)(
    x
)


# ------------------------------------------------------------
# Regression output
# ------------------------------------------------------------

outputs = Dense(
    1
)(
    x
)


transformer_model = Model(
    inputs=inputs,
    outputs=outputs
)


transformer_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


transformer_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 60, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 60, 64)    │      2,112 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 60, 64)    │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 60, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 60, 64)    │          0 │ add[0][0],        │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 60, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 60, 128)   │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 60, 64)    │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │         33 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 37,697 (147.25 KB)

 Trainable params: 37,697 (147.25 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
transformer_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


transformer_history = transformer_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        transformer_early_stopping
    ],

    verbose=1
)

Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 1.7137 - mae: 0.8702 - val_loss: 2.7565 - val_mae: 1.0194
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6426 - mae: 0.8260 - val_loss: 2.7548 - val_mae: 1.0246
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6306 - mae: 0.8165 - val_loss: 2.7510 - val_mae: 1.0187
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6197 - mae: 0.8166 - val_loss: 2.7541 - val_mae: 1.0224
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6183 - mae: 0.8152 - val_loss: 2.7760 - val_mae: 1.0194
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6175 - mae: 0.8153 - val_loss: 2.7599 - val_mae: 1.0184
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6141 - mae: 0.8119 - val_loss: 2.7613 - val_mae: 1.0202
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 1.6176 - mae: 0.8159 - val_loss: 2.7644 - val_mae: 1.0229
Epoch 9/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - lo

In [50]:
transformer_predictions = (
    transformer_model
    .predict(
        X_test
    )
    .flatten()
)


transformer_mae = mean_absolute_error(
    y_test,
    transformer_predictions
)


transformer_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        transformer_predictions
    )
)


transformer_direction_accuracy = (
    directional_accuracy(
        y_test,
        transformer_predictions
    )
)


print(
    "Transformer MAE:",
    transformer_mae
)

print(
    "Transformer RMSE:",
    transformer_rmse
)

print(
    "Transformer Direction Accuracy:",
    transformer_direction_accuracy
)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Transformer MAE: 1.3053313493728638
Transformer RMSE: 2.2707881631354208
Transformer Direction Accuracy: 0.4445945945945946


In [51]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Zero Return",
        "LSTM",
        "BiLSTM",
        "Transformer"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae,
        bilstm_mae,
        transformer_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse,
        bilstm_rmse,
        transformer_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy,
        bilstm_direction_accuracy,
        transformer_direction_accuracy
    ]
})


display(
    comparison
)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.297797,2.262633,0.547297
1,LSTM,1.294408,2.272168,0.564865
2,BiLSTM,1.316356,2.313081,0.502703
3,Transformer,1.305331,2.270788,0.444595


In [52]:
# ============================================================
# 10. TTAO IMPORT
# ============================================================

import sys
from pathlib import Path

from tensorflow.keras.optimizers import Adam


TTAO_ROOT = Path(
    r"C:\Users\bilal\Desktop\ProjetPython\ttao"
)

if str(TTAO_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(TTAO_ROOT)
    )


from ttao import (
    HyperparameterSpec,
    SearchSpace,
    compute_hpo_query_budget,
    optimize_hpo,
)


print("TTAO imported successfully.")


# ============================================================
# 11. TTAO SEARCH SPACE
# ============================================================

tta_space = SearchSpace([

    HyperparameterSpec(
        "lstm_units",
        "integer",
        32,
        128,
        step=32
    ),

    HyperparameterSpec(
        "dense_units",
        "integer",
        16,
        64,
        step=16
    ),

    HyperparameterSpec(
        "dropout",
        "continuous",
        0.1,
        0.5
    ),

    HyperparameterSpec(
        "learning_rate",
        "continuous",
        1e-4,
        5e-3,
        scale="log"
    ),

    HyperparameterSpec(
        "batch_size",
        "categorical",
        choices=(
            16,
            32,
            64
        )
    )
])


# ============================================================
# 12. TTAO OBJECTIVE FUNCTION
# ============================================================

def objective(params):

    # Libérer le modèle précédent
    tf.keras.backend.clear_session()

    # Reproductibilité
    tf.random.set_seed(42)
    np.random.seed(42)


    # Hyperparamètres proposés par TTAO
    lstm_units = int(
        params["lstm_units"]
    )

    dense_units = int(
        params["dense_units"]
    )

    dropout = float(
        params["dropout"]
    )

    learning_rate = float(
        params["learning_rate"]
    )

    batch_size = int(
        params["batch_size"]
    )


    # Construire le modèle
    model = Sequential([

        LSTM(
            lstm_units,
            input_shape=(
                WINDOW_SIZE,
                n_features
            )
        ),

        Dropout(
            dropout
        ),

        Dense(
            dense_units,
            activation="relu"
        ),

        Dense(1)
    ])


    # Compilation
    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )


    # Early stopping
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )


    # Entraînement
    model.fit(
        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=40,

        batch_size=batch_size,

        callbacks=[
            early_stopping
        ],

        verbose=0
    )


    # Prédiction VALIDATION uniquement
    val_predictions = (
        model
        .predict(
            X_val,
            verbose=0
        )
        .flatten()
    )


    # Fitness minimisée par TTAO
    val_rmse = np.sqrt(
        mean_squared_error(
            y_val,
            val_predictions
        )
    )


    print(
        "Params:",
        params,
        "| Validation RMSE:",
        round(
            val_rmse,
            6
        )
    )


    return float(
        val_rmse
    )


# ============================================================
# 13. TEST DE L'OBJECTIVE
# ============================================================

test_params = {
    "lstm_units": 64,
    "dense_units": 32,
    "dropout": 0.2,
    "learning_rate": 0.001,
    "batch_size": 32
}


test_fitness = objective(
    test_params
)


print(
    "\nTest Validation RMSE:",
    test_fitness
)


# ============================================================
# 14. TTAO OPTIMIZATION
# ============================================================

NUM_TRIANGLES = 2
MAX_ITERATIONS = 5


tta_result = optimize_hpo(

    objective_function=objective,

    search_space=tta_space,

    triangle_size_method="paper",

    alpha_method="paper",

    max_iterations=MAX_ITERATIONS,

    num_triangle=NUM_TRIANGLES,

    max_evaluations=compute_hpo_query_budget(
        NUM_TRIANGLES,
        MAX_ITERATIONS
    ),

    seed=42,

    boundary_method="reflect"
)


# ============================================================
# 15. BEST RESULT
# ============================================================

print(
    "\nBest parameters:"
)

print(
    tta_result.best_params
)


print(
    "\nBest validation RMSE:"
)

print(
    tta_result.best_fitness
)

TTAO imported successfully.



c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 64, 'dense_units': 32, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 32} | Validation RMSE: 1.653804

Test Validation RMSE: 1.6538039805165212


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 128, 'dense_units': 48, 'dropout': 0.13767093915505982, 'learning_rate': 0.004545201182466366, 'batch_size': 32} | Validation RMSE: 1.654245


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


... (97 outputs left out)

c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 64, 'dense_units': 48, 'dropout': 0.3120666918974577, 'learning_rate': 0.00020060999512363438, 'batch_size': 16} | Validation RMSE: 1.653138


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Params: {'lstm_units': 96, 'dense_units': 64, 'dropout': 0.3828875521137861, 'learning_rate': 0.000534876348073467, 'batch_size': 16} | Validation RMSE: 1.653611

Best parameters:
{'lstm_units': 64, 'dense_units': 16, 'dropout': 0.19441878517721417, 'learning_rate': 0.00014435236110703763, 'batch_size': 32}

Best validation RMSE:
1.6420732917597998


In [53]:
# ============================================================
# 16. FINAL TTAO-LSTM
# ============================================================

best_params = tta_result.best_params

tf.keras.backend.clear_session()

tf.random.set_seed(42)
np.random.seed(42)


tta_lstm_model = Sequential([

    LSTM(
        int(best_params["lstm_units"]),
        input_shape=(
            WINDOW_SIZE,
            n_features
        )
    ),

    Dropout(
        float(best_params["dropout"])
    ),

    Dense(
        int(best_params["dense_units"]),
        activation="relu"
    ),

    Dense(1)
])


tta_lstm_model.compile(
    optimizer=Adam(
        learning_rate=float(
            best_params["learning_rate"]
        )
    ),
    loss="mse",
    metrics=["mae"]
)


tta_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


tta_history = tta_lstm_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=int(
        best_params["batch_size"]
    ),

    callbacks=[
        tta_early_stopping
    ],

    verbose=1
)

Epoch 1/100


c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 1.6636 - mae: 0.8366 - val_loss: 2.7310 - val_mae: 1.0195
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.6388 - mae: 0.8240 - val_loss: 2.7328 - val_mae: 1.0185
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 1.6237 - mae: 0.8222 - val_loss: 2.7370 - val_mae: 1.0191
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.6171 - mae: 0.8212 - val_loss: 2.7303 - val_mae: 1.0170
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.6076 - mae: 0.8162 - val_loss: 2.7247 - val_mae: 1.0162
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.5986 - mae: 0.8159 - val_loss: 2.7199 - val_mae: 1.0145
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 1.5936 - mae: 0.8149 - val_loss: 2.7282 - val_mae: 1.0149
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.5810 - mae: 0.8117 - val_loss: 2.7200 - val_mae: 1.0134
Epoch 9/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 1.5795 -

In [54]:
# ============================================================
# 17. FINAL TEST EVALUATION
# ============================================================

tta_predictions = (
    tta_lstm_model
    .predict(
        X_test
    )
    .flatten()
)


tta_mae = mean_absolute_error(
    y_test,
    tta_predictions
)


tta_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tta_predictions
    )
)


tta_direction_accuracy = (
    directional_accuracy(
        y_test,
        tta_predictions
    )
)


print(
    "TTAO-LSTM MAE:",
    tta_mae
)

print(
    "TTAO-LSTM RMSE:",
    tta_rmse
)

print(
    "TTAO-LSTM Directional Accuracy:",
    tta_direction_accuracy
)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
TTAO-LSTM MAE: 1.3041009902954102
TTAO-LSTM RMSE: 2.271577998796048
TTAO-LSTM Directional Accuracy: 0.5351351351351351


In [55]:
comparison_final = pd.DataFrame({

    "Model": [
        "Naive Zero Return",
        "LSTM",
        "BiLSTM",
        "Transformer",
        "TTAO-LSTM"
    ],

    "MAE": [
        baseline_mae,
        lstm_mae,
        bilstm_mae,
        transformer_mae,
        tta_mae
    ],

    "RMSE": [
        baseline_rmse,
        lstm_rmse,
        bilstm_rmse,
        transformer_rmse,
        tta_rmse
    ],

    "Directional Accuracy": [
        baseline_direction_accuracy,
        lstm_direction_accuracy,
        bilstm_direction_accuracy,
        transformer_direction_accuracy,
        tta_direction_accuracy
    ]
})


display(
    comparison_final
)

,Model,MAE,RMSE,Directional Accuracy
0,Naive Zero Return,1.297797,2.262633,0.547297
1,LSTM,1.294408,2.272168,0.564865
2,BiLSTM,1.316356,2.313081,0.502703
3,Transformer,1.305331,2.270788,0.444595
4,TTAO-LSTM,1.304101,2.271578,0.535135


In [57]:
# ============================================================
# SAVE FINAL FORECAST MODEL
# ============================================================

import joblib
from pathlib import Path

forecast_dir = Path(
    "../models/forecast"
)

forecast_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Modèle LSTM retenu
lstm_model.save(
    forecast_dir / "lstm_forecast.keras"
)

# Scaler ajusté uniquement sur le train
joblib.dump(
    scaler,
    forecast_dir / "feature_scaler.pkl"
)

# Ordre exact des features
joblib.dump(
    feature_columns,
    forecast_dir / "feature_columns.pkl"
)

# Taille de fenêtre
joblib.dump(
    WINDOW_SIZE,
    forecast_dir / "window_size.pkl"
)

print("Forecast artifacts saved.")

Forecast artifacts saved.
